In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "ic" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"


In [69]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import spearmanr, chi2_contingency, kruskal


In [70]:
df1 = pd.read_csv(
    DATA_DIR / "dados_gerais_bronze1.csv",index_col=0,
    
)

In [71]:
# ---- 1) Build a robust text key (so small spacing/case differences don't break grouping)
df1 = df1.copy()
df1["text_key"] = (
    df1["full_text"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

# Ensure retweet_count numeric
df1["retweet_count"] = pd.to_numeric(df1["retweet_count"], errors="coerce").fillna(0)

# ---- 2) Compare "times tweet appears in dataframe" vs retweet_count
# If retweets were rows, a common expectation would be:
# appearances ~= retweet_count + 1  (1 original + N retweet rows)
g = (
    df1.groupby("text_key", as_index=False)
       .agg(
           appearances=("text_key", "size"),
           max_retweet_count=("retweet_count", "max"),
           mean_retweet_count=("retweet_count", "mean"),
           min_retweet_count=("retweet_count", "min"),
           sample_text=("full_text", "first"),
       )
)

g["expected_if_rows_are_retweets"] = g["max_retweet_count"] + 1
g["diff_vs_expected"] = g["appearances"] - g["expected_if_rows_are_retweets"]
g["exact_match"] = g["diff_vs_expected"] == 0
g["close_match_abs_1"] = g["diff_vs_expected"].abs() <= 1

# ---- 3) Global diagnostics
summary = {
    "unique_tweets_by_text_key": int(len(g)),
    "exact_match_count": int(g["exact_match"].sum()),
    "exact_match_pct": float((g["exact_match"].mean() * 100).round(2)),
    "close_match_abs_1_count": int(g["close_match_abs_1"].sum()),
    "close_match_abs_1_pct": float((g["close_match_abs_1"].mean() * 100).round(2)),
    "pearson_corr_appearances_vs_max_rt": float(g["appearances"].corr(g["max_retweet_count"], method="pearson")),
    "spearman_corr_appearances_vs_max_rt": float(g["appearances"].corr(g["max_retweet_count"], method="spearman")),
}

print("=== Retweet-as-row check summary ===")
for k, v in summary.items():
    print(f"{k}: {v}")

# ---- 4) Inspect strongest mismatches
print("\n=== Top mismatches (largest absolute difference) ===")
display(
    g.loc[:, ["appearances", "max_retweet_count", "expected_if_rows_are_retweets",
              "diff_vs_expected", "sample_text"]]
     .assign(abs_diff=lambda x: x["diff_vs_expected"].abs())
     .sort_values("abs_diff", ascending=False)
     .head(20)
)

# ---- 5) Optional: simple conclusion flag
if summary["exact_match_pct"] < 20:
    print("\nConclusion: retweet_count is probably NOT the number of duplicate tweet rows in this dataframe.")
else:
    print("\nConclusion: there may be partial alignment, inspect mismatches before concluding.")

=== Retweet-as-row check summary ===
unique_tweets_by_text_key: 232007
exact_match_count: 195394
exact_match_pct: 84.22
close_match_abs_1_count: 214504
close_match_abs_1_pct: 92.46
pearson_corr_appearances_vs_max_rt: 0.011621061160097658
spearman_corr_appearances_vs_max_rt: 0.06170370938240214

=== Top mismatches (largest absolute difference) ===


,appearances,max_retweet_count,expected_if_rows_are_retweets,diff_vs_expected,sample_text,abs_diff
163748,108,51557,51558,-51450,LULA NO PRIMEIRO TURNO,51450
127030,5,42070,42071,-42066,Amor à pátria é o c@ralh%! \n\nAMOR AO POVO. \...,42066
226822,1,20521,20522,-20521,|￣￣￣￣￣￣￣￣￣￣￣|\n ESTUDANTE DE \n ...,20521
189238,4,17049,17050,-17046,Na hora de votar não esqueça que Bolsonaro cor...,17046
119908,1,15667,15668,-15667,a gente: mas eu não boto uma camisa vermelha p...,15667
126414,1,14365,14366,-14365,Amanhã é Lula mas daqui a 4 anos será o Léo Pe...,14365
147384,1,12796,12797,-12796,eu decidi pelo voto útil Lula 13 e acredito q ...,12796
204817,1,11821,11822,-11821,"pra quem chorou com o golpe na DILMA, com a pr...",11821
180333,1,11803,11804,-11803,lula pulando q nem uma britadeira e o alckmin ...,11803
146219,1,10613,10614,-10613,Estamos combinados: o ex-presidiário ganha nas...,10613



Conclusion: there may be partial alignment, inspect mismatches before concluding.


In [72]:
df2 = pd.read_csv(DATA_DIR / "tweets_brutos_bronze2.csv", index_col=0)

In [73]:
df2.head()

,full_text
0,se o lula ganhar eu quero uma roda de beijo co...
1,@ixslorena LULA PRESIDENTE HOJE
2,mãe morrendo de alegria na carreata do Lula
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...
4,@indieoffw O RJ elegendo castro e juram que va...


In [74]:
len(df2)

270301

In [75]:
df3 = pd.read_parquet(DATA_DIR / "tweets_base_normalizados.parquet",)

In [76]:
df3.head()

,full_text,clean_text
0,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...
1,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE
2,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...
4,@indieoffw O RJ elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...


In [77]:
len(df3)

192954

In [18]:
df4 = pd.read_parquet(DATA_DIR / "tweets_192k_labeled.parquet",)

In [19]:
df4.head()

,full_text,clean_text,unsupervised_sentiment
0,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...,1.0
1,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE,0.0
2,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula,0.0
3,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...,0.0
4,@indieoffw O RJ elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...,0.0


In [20]:
# 1) Validate minimum columns
required_df = {"user_id", "full_text"}
required_df2 = {"full_text"}
required_df3_any = {"full_text"}  # + clean_text-like column checked below

In [21]:
for name, frame, req in [("df", df, required_df), ("df2", df2, required_df2), ("df3", df3, required_df3_any)]:
    missing = req - set(frame.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")

In [22]:
d1 = df.copy().reset_index(drop=True)
d2 = df2.copy()
d3 = df3.copy()

In [23]:
def text_key(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .str.replace(r"\s+", " ", regex=True)
    )

In [24]:
d1["_text_key"] = text_key(d1["full_text"])
d2["_text_key"] = text_key(d2["full_text"])
d3["_text_key"] = text_key(d3["full_text"])

In [25]:
# 4) Create tweet_id (if not already present)
if "tweet_id" not in d1.columns:
    d1["tweet_id"] = "tw_" + (d1.index + 1).astype(str).str.zfill(9)


In [29]:
# 5) Prepare safe merge tables (avoid row explosion)
d2_stats = d2.groupby("_text_key", as_index=False).size().rename(columns={"size": "count_in_df2"})
d3_map = d3[["_text_key", 'clean_text']].drop_duplicates(subset=["_text_key"]).rename(columns={'clean_text': "clean_text"})

In [32]:
# 6) Build master table
master = (
    d1
    .merge(d2_stats, on="_text_key", how="left")
    .merge(d3_map, on="_text_key", how="left")
)

master["count_in_df2"] = master["count_in_df2"].fillna(0).astype(int)
master["in_df2"] = master["count_in_df2"] > 0

In [33]:
priority = ["tweet_id", "user_id", "screen_name", "location", "created_at", "full_text", "clean_text", "in_df2", "count_in_df2"]
priority = [c for c in priority if c in master.columns]
master = master[priority + [c for c in master.columns if c not in priority]]


In [34]:
# 8) Quick checks
print("master rows:", len(master), "| expected:", len(df))
print("tweet_id unique:", master["tweet_id"].nunique(), "| expected:", len(master))
print("missing clean_text:", int(master["clean_text"].isna().sum()))

master rows: 270301 | expected: 270301
tweet_id unique: 270301 | expected: 270301
missing clean_text: 59808


In [35]:
master.head(3)

,tweet_id,user_id,screen_name,location,created_at,full_text,clean_text,in_df2,count_in_df2,name,followers_count,retweet_count,favorite_count,lang,hashtags,source,_text_key
0,tw_000000001,772193576,pauloisfab,"Sao Paulo, Brazil",Sat Oct 01 23:59:59 +0000 2022,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...,True,1,paulo rouge,439,0,0,pt,[],Twitter for iPhone,se o lula ganhar eu quero uma roda de beijo co...
1,tw_000000002,1546155973051662336,jorgehen_pr,NaN,Sat Oct 01 23:59:59 +0000 2022,@ixslorena LULA PRESIDENTE HOJE,@USER LULA PRESIDENTE HOJE,True,1,quem me conhece sabe 🚩,27,0,0,pt,[],Twitter for Android,@ixslorena lula presidente hoje
2,tw_000000003,1515570566736003072,aanandaaluz,pqp,Sat Oct 01 23:59:59 +0000 2022,mãe morrendo de alegria na carreata do Lula,mãe morrendo de alegria na carreata do Lula,True,1,Ananda,14,0,3,pt,[],Twitter for Android,mãe morrendo de alegria na carreata do lula


In [52]:
def cramers_v_bias_corrected(confusion_matrix: pd.DataFrame) -> float:
    chi2 = chi2_contingency(confusion_matrix, correction=False)[0]
    n = confusion_matrix.to_numpy().sum()
    if n == 0:
        return np.nan
    r, k = confusion_matrix.shape
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / max(n - 1, 1))
    rcorr = r - ((r - 1) ** 2) / max(n - 1, 1)
    kcorr = k - ((k - 1) ** 2) / max(n - 1, 1)
    denom = max(min(kcorr - 1, rcorr - 1), 1e-12)
    return np.sqrt(phi2corr / denom)

In [53]:
def _reduce_cat(s: pd.Series, top_n: int = 20) -> pd.Series:
    s = s.astype("object").where(~pd.isna(s), "__MISSING__")
    keep = set(s.value_counts(dropna=False).head(top_n).index)
    return s.where(s.isin(keep), "__OTHER__")

In [54]:
def relation_scan(
    df: pd.DataFrame,
    force_cats=("location",),
    drop_cols=("full_text", "clean_text", "_text_key"),
    top_n_categories=20,
    min_n=30,
    min_group_size=20
):
    data = df.copy()
    data = data[[c for c in data.columns if c not in set(drop_cols)]]

    # Skip ID-like columns by default (unless forced)
    id_like = [c for c in data.columns if c.lower().endswith("_id") or c.lower() in {"id", "tweet_id"}]
    id_like = [c for c in id_like if c not in force_cats]

    numeric, categorical, skipped = [], [], []

    for c in data.columns:
        if c in id_like:
            skipped.append((c, "id_like"))
            continue

        s = data[c]
        if pd.api.types.is_bool_dtype(s):
            data[c] = s.astype(int)
            numeric.append(c)
        elif pd.api.types.is_numeric_dtype(s):
            if s.nunique(dropna=True) >= 2:
                numeric.append(c)
            else:
                skipped.append((c, "constant_numeric"))
        else:
            if s.nunique(dropna=True) >= 2 or c in force_cats:
                categorical.append(c)
            else:
                skipped.append((c, "constant_categorical"))

    for fc in force_cats:
        if fc in data.columns and fc not in categorical and fc not in numeric:
            categorical.append(fc)

    cat_reduced = {c: _reduce_cat(data[c], top_n=top_n_categories) for c in categorical}
    results = []

    # num-num
    for a, b in combinations(numeric, 2):
        tmp = data[[a, b]].dropna()
        if len(tmp) < min_n or tmp[a].nunique() < 2 or tmp[b].nunique() < 2:
            continue
        rho, p = spearmanr(tmp[a], tmp[b])
        if np.isnan(rho):
            continue
        results.append({
            "var1": a, "var2": b, "type": "num-num",
            "metric": "spearman_rho",
            "strength": float(rho), "abs_strength": abs(float(rho)),
            "p_value": float(p), "n": len(tmp)
        })

    # cat-cat
    for a, b in combinations(categorical, 2):
        tmp = pd.DataFrame({a: cat_reduced[a], b: cat_reduced[b]}).dropna()
        if len(tmp) < min_n:
            continue
        ct = pd.crosstab(tmp[a], tmp[b])
        if ct.shape[0] < 2 or ct.shape[1] < 2:
            continue
        _, p, _, _ = chi2_contingency(ct, correction=False)
        v = cramers_v_bias_corrected(ct)
        results.append({
            "var1": a, "var2": b, "type": "cat-cat",
            "metric": "cramers_v",
            "strength": float(v), "abs_strength": abs(float(v)),
            "p_value": float(p), "n": len(tmp)
        })

    # num-cat
    for ncol in numeric:
        for ccol in categorical:
            tmp = pd.DataFrame({ncol: data[ncol], ccol: cat_reduced[ccol]}).dropna()
            if len(tmp) < min_n:
                continue

            group_sizes = tmp[ccol].value_counts()
            valid_levels = group_sizes[group_sizes >= min_group_size].index
            tmp = tmp[tmp[ccol].isin(valid_levels)]
            if tmp[ccol].nunique() < 2:
                continue

            if tmp[ncol].nunique() < 2:
                continue

            groups = [g[ncol].values for _, g in tmp.groupby(ccol)]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) < 2:
                continue

            all_vals = np.concatenate(groups)
            if pd.Series(all_vals).nunique() < 2:
                continue

            try:
                H, p = kruskal(*groups)
            except ValueError:
                continue

            n, k = len(tmp), len(groups)
            eps2 = max(0.0, (H - k + 1) / max(n - k, 1))
            results.append({
                "var1": ncol, "var2": ccol, "type": "num-cat",
                "metric": "kruskal_eps2",
                "strength": float(eps2), "abs_strength": abs(float(eps2)),
                "p_value": float(p), "n": n
            })

    out = pd.DataFrame(results)
    if not out.empty:
        out = out.sort_values("p_value").reset_index(drop=True)
        m = len(out)
        out["rank"] = np.arange(1, m + 1)
        out["q_value_bh"] = (out["p_value"] * m / out["rank"]).clip(upper=1.0)
        out["significant_0_05"] = out["q_value_bh"] < 0.05
        out = out.sort_values(
            ["significant_0_05", "abs_strength", "q_value_bh"],
            ascending=[False, False, True]
        ).reset_index(drop=True)

    coverage = {
        "numeric_used": numeric,
        "categorical_used": categorical,
        "skipped": pd.DataFrame(skipped, columns=["column", "reason"]),
        "type_counts": out["type"].value_counts().to_dict() if not out.empty else {}
    }

    return out, coverage



In [55]:
scan, coverage = relation_scan(master, force_cats=("location", "lang", "source", "hashtags"))
display(pd.Series(coverage["type_counts"], name="count"))
display(pd.DataFrame({"numeric_used": pd.Series(coverage["numeric_used"]),
                      "categorical_used": pd.Series(coverage["categorical_used"])}))
display(coverage["skipped"])
display(scan.head(50))

num-cat    24
cat-cat    15
num-num     6
Name: count, dtype: int64

,numeric_used,categorical_used
0,in_df2,screen_name
1,count_in_df2,location
2,followers_count,created_at
3,retweet_count,name
4,favorite_count,lang
5,NaN,hashtags
6,NaN,source


,column,reason
0,tweet_id,id_like
1,user_id,id_like


,var1,var2,type,metric,strength,abs_strength,p_value,n,rank,q_value_bh,significant_0_05
0,screen_name,name,cat-cat,cramers_v,0.863259,0.863259,0.000000e+00,270301,22,0.000000e+00,True
1,retweet_count,favorite_count,num-num,spearman_rho,0.490739,0.490739,0.000000e+00,270301,21,0.000000e+00,True
2,followers_count,favorite_count,num-num,spearman_rho,0.349173,0.349173,0.000000e+00,270301,20,0.000000e+00,True
3,followers_count,retweet_count,num-num,spearman_rho,0.269221,0.269221,0.000000e+00,270301,19,0.000000e+00,True
4,screen_name,location,cat-cat,cramers_v,0.162453,0.162453,0.000000e+00,270301,17,0.000000e+00,True
5,location,name,cat-cat,cramers_v,0.141926,0.141926,0.000000e+00,270301,27,0.000000e+00,True
6,count_in_df2,favorite_count,num-num,spearman_rho,-0.133122,0.133122,0.000000e+00,270301,18,0.000000e+00,True
7,count_in_df2,followers_count,num-num,spearman_rho,-0.121935,0.121935,0.000000e+00,270301,1,0.000000e+00,True
8,screen_name,source,cat-cat,cramers_v,0.096210,0.096210,0.000000e+00,270301,24,0.000000e+00,True
9,name,source,cat-cat,cramers_v,0.095282,0.095282,0.000000e+00,270301,14,0.000000e+00,True


In [56]:
scan[(scan["type"]=="num-cat") & (scan["var1"]=="retweet_count") & (scan["var2"]=="location")]


,var1,var2,type,metric,strength,abs_strength,p_value,n,rank,q_value_bh,significant_0_05
30,retweet_count,location,num-cat,kruskal_eps2,0.004983,0.004983,1.448116e-277,270301,29,2.247076e-277,True


In [68]:
# 1) Detect clean text column in df3
clean_col_candidates = ["clean_text", "normalized_text", "text_clean"]
clean_col = next((c for c in clean_col_candidates if c in df3.columns), None)
if clean_col is None:
    raise ValueError(f"df3 must contain one of: {clean_col_candidates}")

# 2) Normalize merge key
def _text_key(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .str.replace(r"\s+", " ", regex=True)
    )

d1 = df1.copy()
d3 = df3.copy()

d1["_text_key"] = _text_key(d1["full_text"])
d3["_text_key"] = _text_key(d3["full_text"])

# 3) Guarantee unique tweets on df3 side (already expected, but safe)
d3_unique = d3.drop_duplicates(subset=["_text_key"]).copy()

# 4) Merge metadata (df1) + normalized text (df3)
master_nodup = d1.merge(
    d3_unique[["_text_key", clean_col]],
    on="_text_key",
    how="inner"   # keep only tweets present in df3 unique base
).rename(columns={clean_col: "clean_text"})

# 5) Enforce one row per unique tweet after merge
master_nodup = (
    master_nodup
    .drop_duplicates(subset=["_text_key"], keep="first")
    .reset_index(drop=True)
)

# 6) Add tweet_id if missing
if "tweet_id" not in master_nodup.columns:
    master_nodup["tweet_id"] = "tw_" + (master_nodup.index + 1).astype(str).str.zfill(9)

# 7) Quick validation
print("rows:", len(master_nodup))
print("unique _text_key:", master_nodup["_text_key"].nunique())
print("duplicates by _text_key:", master_nodup.duplicated(subset=["_text_key"]).sum())

master_nodup.head(50)

rows: 192399
unique _text_key: 192399
duplicates by _text_key: 0


,user_id,screen_name,name,location,followers_count,created_at,full_text,retweet_count,favorite_count,lang,hashtags,source,_text_key,clean_text,tweet_id
0,772193576,pauloisfab,paulo rouge,"Sao Paulo, Brazil",439,Sat Oct 01 23:59:59 +0000 2022,se o lula ganhar eu quero uma roda de beijo co...,0,0,pt,[],Twitter for iPhone,se o lula ganhar eu quero uma roda de beijo co...,se o lula ganhar eu quero uma roda de beijo co...,tw_000000001
1,1546155973051662336,jorgehen_pr,quem me conhece sabe 🚩,NaN,27,Sat Oct 01 23:59:59 +0000 2022,@ixslorena LULA PRESIDENTE HOJE,0,0,pt,[],Twitter for Android,@ixslorena lula presidente hoje,@USER LULA PRESIDENTE HOJE,tw_000000002
2,1515570566736003072,aanandaaluz,Ananda,pqp,14,Sat Oct 01 23:59:59 +0000 2022,mãe morrendo de alegria na carreata do Lula,0,3,pt,[],Twitter for Android,mãe morrendo de alegria na carreata do lula,mãe morrendo de alegria na carreata do Lula,tw_000000003
3,1449859889589899264,lyeroses,sasa៹ 13 ⭐🚩,only blink,4973,Sat Oct 01 23:59:59 +0000 2022,@trajaza AVISA QUE MEU PAI LULA VAI GANHAR PRI...,0,3,pt,[],Twitter for Android,@trajaza avisa que meu pai lula vai ganhar pri...,@USER AVISA QUE MEU PAI LULA VAI GANHAR PRIMEI...,tw_000000004
4,1304397938131599361,HIGH_G_LUIZ,łu¡z 🪐,IZ*ONE,422,Sat Oct 01 23:59:59 +0000 2022,@indieoffw O RJ elegendo castro e juram que va...,0,0,pt,[],Twitter for Android,@indieoffw o rj elegendo castro e juram que va...,@USER O RJ elegendo castro e juram que vai dar...,tw_000000005
5,1564043448172306433,analourdesfn,Ana Lourdes,"Juazeiro do Norte, Brasil",53,Sat Oct 01 23:59:59 +0000 2022,LULA PRESIDENTE AMANHÃ 1️⃣3️⃣🚩❤ https://t.co/T...,1,4,pt,[],Twitter for Android,lula presidente amanhã 1️⃣3️⃣🚩❤ https://t.co/t...,LULA PRESIDENTE AMANHÃ 1️⃣3️⃣ :bandeira_triang...,tw_000000006
6,381204823,alex_paranaense,alex 1️⃣2️⃣,NaN,322,Sat Oct 01 23:59:59 +0000 2022,@DiRamaciotti Bora de Ciro está em terceiro lu...,0,3,pt,[],Twitter for Android,@diramaciotti bora de ciro está em terceiro lu...,@USER Bora de Ciro está em terceiro lugar e se...,tw_000000007
7,1262950474648563713,HellendaRochaV1,hellenzinha ⭐️,Santa Catarina,372,Sat Oct 01 23:59:59 +0000 2022,@denisoon_ Meus primos que nunca trabalharam n...,0,0,pt,[],Twitter for iPhone,@denisoon_ meus primos que nunca trabalharam n...,@USER Meus primos que nunca trabalharam na vid...,tw_000000008
8,1087515836955443200,quarentener_,eu sou tao galera,"Curitiba, Brazil",147,Sat Oct 01 23:59:59 +0000 2022,Se o Lula realmente ganhar no primeiro turno e...,0,4,pt,[],Twitter for Android,se o lula realmente ganhar no primeiro turno e...,Se o Lula realmente ganhar no primeiro turno e...,tw_000000009
9,1084139131385626627,cj_mayanasilva,Mayana Silva,NaN,22,Sat Oct 01 23:59:59 +0000 2022,@GuilhermeBoulos Lula 13❤️❤️❤️,0,0,pt,[],Twitter for iPhone,@guilhermeboulos lula 13❤️❤️❤️,@USER Lula 13 :coração_vermelho: :coração_verm...,tw_000000010
